# ECG Datasets Preview (afdb, PTB-XL, Chapman-Shaoxing)

Second data notebook. Previews the other three datasets used in the thesis.

- afdb: long-term ECG with atrial fibrillation episodes
- PTB-XL: short 12-lead clinical ECGs
- Chapman-Shaoxing: large 12-lead hospital dataset


In [ ]:
# packages
%pip install -q wfdb neurokit2 pandas matplotlib

## 1. MIT-BIH Atrial Fibrillation Database (afdb)

25 long-term recordings (~10 h each, 250 Hz). The annotation file marks
AF episodes alongside normal rhythm.


In [ ]:
# download one record (~35 MB)
import wfdb, os
os.makedirs('afdb', exist_ok=True)
wfdb.dl_database('afdb', dl_dir='afdb', records=['04015'])
print(sorted(os.listdir('afdb')))

In [ ]:
# patient info + recording length
rec = wfdb.rdrecord('afdb/04015')
print('leads:', rec.sig_name, '| fs:', rec.fs, 'Hz')
print('length: %.1f hours' % (rec.sig_len / rec.fs / 3600))
for c in rec.comments: print(' ', c)

In [ ]:
# AF vs normal rhythm over the whole recording
ann = wfdb.rdann('afdb/04015', 'atr')
changes = [(s, n) for s, n in zip(ann.sample, ann.aux_note) if n in ('(AFIB', '(N')]
total = rec.sig_len
af_samples = 0
for i in range(len(changes) - 1):
    if changes[i][1] == '(AFIB':
        af_samples += changes[i+1][0] - changes[i][0]
print('AF time: %.1f%% of the recording' % (100*af_samples/total))
print('rhythm switches:', len(changes))

In [ ]:
# plot 60 s around the first AF episode (AF part shaded red)
import numpy as np, matplotlib.pyplot as plt
fs = rec.fs
sig = rec.p_signal[:, 0]
first_af = next(s for s, n in changes if n == '(AFIB')
start = max(0, first_af - 10*fs)
end = start + 60*fs
t = np.arange(start, end) / fs / 60
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, sig[start:end], lw=0.6, color='tab:blue')
for i in range(len(changes) - 1):
    a, b = changes[i][0], changes[i+1][0]
    if changes[i][1] == '(AFIB' and b > start and a < end:
        ax.axvspan(max(a, start)/fs/60, min(b, end)/fs/60, color='red', alpha=0.15)
ax.set_xlabel('time (min)'); ax.set_ylabel('mV'); ax.grid(alpha=0.3)
ax.set_title('record 04015, ECG1 lead - red = AF episode')
plt.show()

## 2. PTB-XL

21,799 short 10 s 12-lead ECGs from 18,869 patients (2019, Germany).
Labels are diagnoses (SNOMED codes) in ptbxl_database.csv.


In [ ]:
# download the metadata table + two records (small, 100 Hz version)
import urllib.request, os
base = 'https://physionet.org/files/ptb-xl/1.0.3/'
os.makedirs('ptbxl/records100/00000', exist_ok=True)
for f in ['ptbxl_database.csv']:
    urllib.request.urlretrieve(base + f, 'ptbxl/' + f)
for ecg in ['00001', '00002']:
    for ext in ['.dat', '.hea']:
        urllib.request.urlretrieve(base + f'records100/00000/{ecg}_lr{ext}',
                                    f'ptbxl/records100/00000/{ecg}_lr{ext}')
print('downloaded')

In [ ]:
# metadata: first rows + patient age/sex
import pandas as pd
df = pd.read_csv('ptbxl/ptbxl_database.csv', index_col='ecg_id')
print('total records:', len(df))
df[['age', 'sex', 'height', 'weight', 'device']].head()

In [ ]:
# most common diagnoses
codes = ['NORM', 'AFIB', 'LVH', 'IRBBB', 'PAC', 'PVC', 'CRBBB', 'LBBB', 'STTC']
counts = {c: df['scp_codes'].str.contains(c).sum() for c in codes}
plt.figure(figsize=(7, 3))
plt.bar(counts.keys(), counts.values(), color='tab:blue')
plt.xticks(rotation=45); plt.ylabel('records'); plt.grid(alpha=0.3, axis='y')
plt.title('diagnosis codes across PTB-XL (full table)')
plt.show()

In [ ]:
# plot one full 12-lead ECG (10 s)
rec2 = wfdb.rdrecord('ptbxl/records100/00000/00001_lr')
row = df.loc[int('00001')]
print('patient: %d y/o %s | report: %s' % (row.age,
      'male' if row.sex == 1 else 'female', row.report[:80]))
t = np.arange(rec2.sig_len) / rec2.fs
fig, axes = plt.subplots(6, 2, figsize=(12, 9), sharex=True)
for ax, ch in zip(axes.flat, rec2.sig_name):
    ax.plot(t, rec2.p_signal[:, rec2.sig_name.index(ch)], lw=0.6)
    ax.set_title(ch, fontsize=9); ax.grid(alpha=0.3)
fig.suptitle('PTB-XL record 00001 (100 Hz)')
plt.tight_layout(); plt.show()

## 3. Chapman-Shaoxing

45,152 patients, 10 s 12-lead ECGs (Chinese/Japanese hospital data).
Files live under WFDBRecords/, diagnosis is in the .hea comments and
mapped via ConditionNames_SNOMED-CT.csv.


In [ ]:
# download the condition table + one record
import urllib.request, os
base = 'https://physionet.org/files/ecg-arrhythmia/1.0.0/'
os.makedirs('chapman/WFDBRecords/01/010', exist_ok=True)
urllib.request.urlretrieve(base + 'ConditionNames_SNOMED-CT.csv',
                           'chapman/ConditionNames_SNOMED-CT.csv')
for ext in ['.mat', '.hea']:  # chapman uses .mat signal files
    urllib.request.urlretrieve(base + f'WFDBRecords/01/010/JS00005{ext}',
                                f'chapman/WFDBRecords/01/010/JS00005{ext}')
print('downloaded')

In [ ]:
# condition names table
cond = pd.read_csv('chapman/ConditionNames_SNOMED-CT.csv')
print(cond.head(12))

In [ ]:
# plot the record + its diagnosis from the header
rec3 = wfdb.rdrecord('chapman/WFDBRecords/01/010/JS00005')
print('fs:', rec3.fs, '| duration: %.1f s' % (rec3.sig_len / rec3.fs))
for c in rec3.comments: print(' ', c)
t = np.arange(rec3.sig_len) / rec3.fs
fig, axes = plt.subplots(6, 2, figsize=(12, 9), sharex=True)
for ax, ch in zip(axes.flat, rec3.sig_name):
    ax.plot(t, rec3.p_signal[:, rec3.sig_name.index(ch)], lw=0.6)
    ax.set_title(ch, fontsize=9); ax.grid(alpha=0.3)
fig.suptitle('Chapman-Shaoxing JS00005 (500 Hz)')
plt.tight_layout(); plt.show()

## Notes

- afdb: good for rhythm-level AF detection experiments
- PTB-XL: good for testing the model on modern 12-lead data
- Chapman-Shaoxing: large scale, for later work

All from PhysioNet; cite the original papers when used.
